In [57]:
# Install PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")

import torch.nn as nn

PyTorch version: 2.9.1


## Self Attention Weights

Given: the input text has been arranged into input vectors by the embedding layer.
Objective: Compute context vector as the combination of all input vectors and attention weights. The weights defines the importance of input tokens relative to the target query token.

Intermediate attention scores (omega) are computed by the dot product of an input query with each input token. The dot product is a measure of similarity between two vectors. Attention scores are not normalized.

The attention weights (alpha) are computed by normalizing the attention scores. The normalized values sum to one and helps prevent large values during optimization.

Attention weights (alpha) are multiplied to the input vecctor by dot product.

In [2]:
# Test input vector
inputs = torch.tensor([
  [0.43, 0.15, 0.89], # Your     (x^1
  [0.55, 0.87, 0.66], # journey  (x^2)
  [0.57, 0.85, 0.64], # starts   (x^3)
  [0.22, 0.58, 0.33], # with     (x^4)
  [0.77, 0.25, 0.10], # one      (x^5)
  [0.05, 0.80, 0.55]  # step     (x^6)
])

In [3]:
# Make the "journey" input token the query
query = inputs[1]

# Compute attention scores by dot product with query for each input 
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

# Compute attention weights by normalizing the attention scores
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Weights sum:", attn_weights_2_tmp.sum())

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Weights sum: tensor(1.0000)


In [4]:
# Improved normalization using softmax
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Weights sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Weights sum: tensor(1.)


In [5]:
# Make the "journey" input token the query
query = inputs[1]

# Compute context vector by multiplying by each input 
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


Generalize the self-attention mechanism

In [6]:
# Compute attenion scores by matrix multiplication 
attn_scores = inputs @ inputs.T

# Compute attention weights by normalization
# In softmax() the dim=1 means all row values sum to 1
attn_weights = torch.softmax(attn_scores, dim=1)
print(attn_weights)

# Check all rows sum to 1
attn_weights.sum(dim=-1)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [7]:
# Compute context vectors
compute_vec = attn_weights @ inputs
print(compute_vec)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## Trainable Attention Weights

Trainable weight matrices are divided into *query*, *key* and *value* that are applied to specific input vectors to create *query*, *key* and *value* vectors.

In [8]:
# Trainable self-attention computes a context
class SelfAttention_v2(nn.Module):
  def __init__(self, d_in, d_out, qkt_bias=False):
    super().__init__()

    # Initialize the random trainable weight matrices
    self.W_query = nn.Linear(d_in, d_out, bias=qkt_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkt_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkt_bias)

  # Forward pass
  def forward(self, x):
    # Compute weight vectors
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    # Compute attention scores
    attn_scores = queries @ keys.T # omega

    # Compute attention weights by normalizing with softmax
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    # Compute context vectors of all inputs
    context_vec = attn_weights @ values
    return context_vec

In [9]:
# Test SelfAttention_v1 class
d_in = inputs.shape[1]
d_out = 2
torch.manual_seed(123)
sa_v1 = SelfAttention_v2(d_in, d_out)
print(sa_v1(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


## Causal Attention

Change the attention weights to ignore future input text which should be hidden during training.

In [10]:
# Test lower triangular matrix
tril_mask = torch.tril(torch.ones(5, 5))  # retain diagonal
print(tril_mask)
tril_mask = torch.tril(torch.ones(5, 5), diagonal=-1) # below the diagonal
print(tril_mask)

# Test upper triangular matrix
triu_mask = torch.triu(torch.ones(5, 5))  # retain diagonal
print(triu_mask)
triu_mask = torch.triu(torch.ones(5, 5), diagonal=1)  # above the diagonal
print(triu_mask)

# Test setting negative infinity mask 
triu_mask = triu_mask.masked_fill(triu_mask.bool(), -torch.inf)
print(triu_mask)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
tensor([[0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.]])
tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])
tensor([[0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0.]])
tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])


In [11]:
# Test regularization technique of zero-setting dropout with default 50% rate. 
# Note how other values are scaled up by 1/(1 - p) to compensate.
dropout = nn.Dropout()
drop_ones = dropout(torch.ones(5, 5))
print(drop_ones)

tensor([[2., 0., 2., 2., 2.],
        [2., 2., 0., 2., 0.],
        [2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0.],
        [2., 0., 2., 2., 2.]])


In [32]:
# Test matrix transpose of dimensions 1 and 2
values = torch.rand(2, 6, 2)
print(values)
print(values.shape)
values = torch.transpose(values, 1, 2)
print(values)
print(values.shape)

tensor([[[0.6042, 0.9836],
         [0.1444, 0.9010],
         [0.9221, 0.9043],
         [0.5713, 0.9546],
         [0.8339, 0.8730],
         [0.4675, 0.1163]],

        [[0.4938, 0.5938],
         [0.1594, 0.2132],
         [0.0206, 0.3247],
         [0.9355, 0.5855],
         [0.4695, 0.5201],
         [0.8118, 0.0585]]])
torch.Size([2, 6, 2])
tensor([[[0.6042, 0.1444, 0.9221, 0.5713, 0.8339, 0.4675],
         [0.9836, 0.9010, 0.9043, 0.9546, 0.8730, 0.1163]],

        [[0.4938, 0.1594, 0.0206, 0.9355, 0.4695, 0.8118],
         [0.5938, 0.2132, 0.3247, 0.5855, 0.5201, 0.0585]]])
torch.Size([2, 2, 6])


In [ ]:
# Causal self-attention mechanism to compute a context vector for specified input
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout=0.0, qkv_bias=False):
    super().__init__()
    self.d_out = d_out

    # Initialize the random trainable weight matrices
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    # Create dropout layer to reduce overfitting - the model becomes less reliant on
    # any specific hidden set of layer units. Used only in training and then disabled.
    # Dropout is longer used by OpenAI GPT.
    self.dropout = nn.Dropout(dropout)

    # Create a causal mask using the upper triangular matrix and register it so it gets.
    # automatically moved to the appropriate device (CPU or GPU) along with the model.
    # This avoids having to redefine the mask layer during iterations of the forward() method.
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))


  # Forward pass computation
  def forward(self, x):
    # Extract dimensions from input
    batch_size, num_tokens, d_in = x.shape

    # Compute weight vectors
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    # Compute attention scores using dot product of queries times keys
    # Transpose dimensions 1 and 2, keeping the batch dimension at the first position (0)
    attn_scores = queries @ keys.transpose(1, 2) # omega

    # Apply causal mask to remove future tokens.
    # It is performed before normalization so that row sums still sum to 1.
    # The negative infinity values are treated by softmax as zero, since e**(-infinity) = 0.
    # Note everything in PyTorch that ends with "_" runs as an inplace operation (ie. not copied).
    attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

    # Compute attention weights by normalizing with softmax, then apply dropout
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # Compute context vectors of all inputs
    context_vec = attn_weights @ values
    return context_vec

In [46]:
# Test batch stack of 2 inputs, 6 input tokens context length, where each token 
# consists of 3-dimenstional embedding vector
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

# Create causal attention process
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length)

# Apply causal attention
context_vecs = ca(batch)
print(context_vecs)
print(f"context_vecs.shape: {context_vecs.shape}")

torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


## Multi-head Attention

Stack multiple single-head attention layers. This divides the attention mechanism into multiple heads, where each operates independently.

The idea behind multi-head attention is for each head to extract different types of information from the input. Each head is initialized with its own weights and learns differently during training.

In [47]:
# Multi-head causal attention mechanism using array of multiple single-head layers.
# Note: the implementation is not efficient single the heads are computed sequentially.
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0, qkv_bias=False):
    super().__init__()

    # Create mutli-head causal layers
    self.heads = nn.ModuleList(
      [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)]
    )
    
  # Forward pass computation
  def forward(self, x):
    # Concatenate each head
    return torch.cat([head(x) for head in self.heads], dim=-1)

In [51]:
# Create causal attention process
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
num_heads = 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, num_heads)

# Apply causal attention
context_vecs = mha(batch)
print(context_vecs)
print(f"context_vecs.shape: {context_vecs.shape}")

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


In [56]:
# Test matrix splitting
values = torch.rand(2, 6, 4)
print(values)
print(values.shape)

values = values.view(2, 6, 2, 2)
print(values)
print(values.shape)

tensor([[[0.4130, 0.5585, 0.1170, 0.5578],
         [0.6681, 0.9275, 0.3443, 0.6800],
         [0.9998, 0.2855, 0.9753, 0.2518],
         [0.7204, 0.6959, 0.6397, 0.8954],
         [0.2979, 0.6314, 0.5028, 0.1239],
         [0.3786, 0.1661, 0.7211, 0.5449]],

        [[0.5490, 0.3483, 0.5024, 0.3445],
         [0.6437, 0.9856, 0.5757, 0.2785],
         [0.1946, 0.5382, 0.1291, 0.1242],
         [0.1746, 0.3302, 0.5370, 0.8443],
         [0.6937, 0.8831, 0.1861, 0.5422],
         [0.0556, 0.7868, 0.6042, 0.9836]]])
torch.Size([2, 6, 4])
tensor([[[[0.4130, 0.5585],
          [0.1170, 0.5578]],

         [[0.6681, 0.9275],
          [0.3443, 0.6800]],

         [[0.9998, 0.2855],
          [0.9753, 0.2518]],

         [[0.7204, 0.6959],
          [0.6397, 0.8954]],

         [[0.2979, 0.6314],
          [0.5028, 0.1239]],

         [[0.3786, 0.1661],
          [0.7211, 0.5449]]],


        [[[0.5490, 0.3483],
          [0.5024, 0.3445]],

         [[0.6437, 0.9856],
          [0.5757, 0.2

In [ ]:
# Multi-head causal attention mechanism using parallelism from matrix multiplication.
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, num_heads, dropout=0.0, qkv_bias=False):
    super().__init__()
    assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads

    # Reduce the projection dimension to match the output dimension 
    self.head_dim = d_out // num_heads

    # Initialize the random trainable weight matrices
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    # Create projection layer to combine outputs
    self.out_proj = nn.Linear(d_out, d_out)

    # Create dropout layer to reduce overfitting - the model becomes less reliant on
    # any specific hidden set of layer units. Used only in training and then disabled.
    # Dropout is longer used by OpenAI GPT.
    self.dropout = nn.Dropout(dropout)

    # Create a causal mask using the upper triangular matrix and register it so it gets.
    # automatically moved to the appropriate device (CPU or GPU) along with the model.
    # This avoids having to redefine the mask layer during iterations of the forward() method.
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

  # Forward pass computation
  def forward(self, x):
    # Extract dimensions from input
    batch_size, num_tokens, d_in = x.shape

    # Compute weight vectors
    # Shape: batch_size, num_tokens, d_out
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    # Reshape the weight vectors so they can be applied to multiple heads by matrix multiplication.
    # The view() call splits the matrix by adding a num_heads dimension, then unroll the last dimension
    # to result in shape: batch_size, num_tokens, num_heads, head_dim. 
    # View() returns a contigous memory object, and is more efficient than reshape(), which makes a copy.
    queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim)

    # Transpose the middle two dimensionss from (num_tokens, num_heads) to (num_heads, num_tokens)
    queries = queries.transpose(1, 2)
    keys = keys.transpose(1, 2)
    values = values.transpose(1, 2)

    # Compute attention scores using dot product of queries times keys
    # Transpose dimensions 2 and 3
    attn_scores = queries @ keys.transpose(2, 3) # omega

    # Apply causal mask to remove future tokens.
    # It is performed before normalization so that row sums still sum to 1.
    # The negative infinity values are treated by softmax as zero, since e**(-infinity) = 0.
    # Note everything in PyTorch that ends with "_" runs as an inplace operation (ie. not copied).
    attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

    # Compute attention weights by normalizing with softmax, then apply dropout
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # Compute context vectors of all inputs
    # Shape: batch_size, num_tokens, num_heads, head_dim
    context_vec = (attn_weights @ values).transpose(1, 2)

    # Combine heads to output dimension (num_heads, head_dim).
    # The contiguous() call is needed since transpose() may have fragmented the tensor in memory.
    context_vec = context_vec.contiguous().view(batch_size, num_tokens, self.d_out)
    context_vec = self.out_proj(context_vec)
    return context_vec

In [58]:
# Create causal attention process
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
num_heads = 2
mha = MultiHeadAttention(d_in, d_out, context_length, num_heads)

# Apply causal attention
context_vecs = mha(batch)
print(context_vecs)
print(f"context_vecs.shape: {context_vecs.shape}")

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
